# Appendix A: Scale Table & Cosmic Octaves

Executable verification code for all calculations in Appendix A.
Open in Jupyter to modify parameters or rerun.

## Section I: Scale Ladder Statistics

Computes the consecutive delta-log values between the 15 fundamental structures,
verifying the average scale jump of ~10^3 and related statistics.

In [ ]:
import math
import statistics

# 15 structures: peer-reviewed radii (m)
radii = {
    'Proton':              8.40e-16,
    'Atomic Orbital (H)':  5.29e-11,
    'Ribosome':            1.10e-8,
    'Bacterium (E. coli)': 1.00e-6,
    'C. elegans':          5.00e-4,
    'Human':               9.00e-1,
    'City':                1.00e3,
    'Earth':               6.37e6,
    'Sun':                 6.96e8,
    'Solar System':        4.50e12,
    'Open Cluster':        4.70e16,
    'Local Bubble':        4.63e18,
    'Milky Way':           5.00e20,
    'Virgo Supercluster':  6.90e23,
    'Observable Universe': 4.40e26,
}

names = list(radii.keys())
log_r = [math.log10(radii[n]) for n in names]

# 14 consecutive Dlog values
delta_log = [log_r[i] - log_r[i-1] for i in range(1, len(log_r))]

print("Structure                    log10(r)   Dlog")
print("-" * 55)
for i, name in enumerate(names):
    dlog_str = f"{delta_log[i-1]:.2f}" if i > 0 else "(base)"
    print(f"{name:28s} {log_r[i]:7.2f}   {dlog_str}")

mean_val  = statistics.mean(delta_log)
med_val   = statistics.median(delta_log)
sd_val    = statistics.stdev(delta_log)   # sample SD (N-1)
min_val   = min(delta_log)
max_val   = max(delta_log)

print(f"\nMean  = {mean_val:.2f}")
print(f"Median = {med_val:.2f}")
print(f"SD     = {sd_val:.2f}  (sample, N-1)")
print(f"Range  = {min_val:.2f} to {max_val:.2f}")

## Section II: City Radius Sensitivity

Tests the effect of using 1 km vs 10 km city radius on the cosmic octave
deviation and scale ladder statistics.

In [ ]:
import math

# Current city radius
city_current = 1e3  # 1 km
log_city_current = math.log10(city_current)  # 3.00

# Alternative city radius
city_alt = 1e4  # 10 km
log_city_alt = math.log10(city_alt)  # 4.00

# Human log radius
log_human = math.log10(0.9)  # -0.046

# Observable Universe log radius
log_obs = 26.64

# Human -> City jump
delta_current = log_city_current - log_human
delta_alt = log_city_alt - log_human
print(f"Human -> City (1 km):  Dlog = {delta_current:.2f}")
print(f"Human -> City (10 km): Dlog = {delta_alt:.2f}")

# City -> Observable Universe octave
octave_current = log_obs - log_city_current
octave_alt = log_obs - log_city_alt
print(f"\nCity -> Obs Universe (1 km):  10^{octave_current:.2f} (dev = {abs(octave_current - 24):.2f})")
print(f"City -> Obs Universe (10 km): 10^{octave_alt:.2f} (dev = {abs(octave_alt - 24):.2f})")

# Impact on average scale jump
print(f"\nWith 10 km city, Human->City Dlog rises from {delta_current:.2f} to {delta_alt:.2f}")
print(f"City->Earth Dlog falls from 3.80 to {6.80 - log_city_alt:.2f}")
print(f"The octave deviation worsens from {abs(octave_current - 24):.2f} to {abs(octave_alt - 24):.2f}")

## Section III: C. elegans Selection Sensitivity

Compares the cosmic octave deviation when using C. elegans, Drosophila,
or tardigrade as the representative multicellular organism.

In [ ]:
import math

# C. elegans: 0.5 mm radius
log_celegans = math.log10(5e-4)  # -3.30

# Drosophila melanogaster: ~1.5 mm body half-length
log_drosophila = math.log10(1.5e-3)  # -2.82

# Tardigrade: ~0.25 mm half-length
log_tardigrade = math.log10(2.5e-4)  # -3.60

# Milky Way
log_mw = 20.70

# Octave calculations
for name, log_r in [("C. elegans", log_celegans), ("Drosophila", log_drosophila), ("Tardigrade", log_tardigrade)]:
    octave = log_mw - log_r
    dev = abs(octave - 24.0)
    print(f"{name:12s} -> Milky Way: 10^{octave:.2f} (deviation {dev:.2f})")

## Section IV: Geometric Center Calculation

Finds the midpoint of the scale ladder and demonstrates that humans
sit below the geometric center, not at it.

In [ ]:
geo_center = (-15.08 + 26.64) / 2
print(f"Geometric center of ladder: log10(r) = {geo_center:.2f}")
print(f"Human position: log10(r) = -0.046")
print(f"Offset from center: {-0.046 - geo_center:.2f} orders of magnitude")
print(f"\nThe ladder is NOT symmetric around humans.")
print(f"Humans sit {abs(-0.046 - geo_center):.1f} orders below the geometric midpoint.")

## Section V: The Headline Permutation Test

The test that produces the manuscript's canonical p = 0.000055 result. This is the same calculation referenced throughout the book (Chapter 11, Chapter 19, master index). Reproduces verbatim from [code/permutation_test.py](https://github.com/Chris-L78/cosmic-octaves-analysis/blob/main/code/permutation_test.py).

**Method:** Permute the 15 measured log-lengths, count strong-match pairs (deviation ≤ 0.2 from 24.0) under each permutation, repeat 200,000 times. Compare against the 3 strong matches observed in the canonical pair assignment.

**Why permutation (not uniform-random Monte Carlo):** Permutation preserves the actual measured values and tests only whether the *labeling* of pairs shows unusual alignment. Uniform-random MC would test a different (weaker) null hypothesis.

**Why 7 fixed pairs (not all 105 pairs):** The cosmic octave hypothesis predicts specific pairings (each rung paired with a structure 10²⁴ larger). Testing all C(15,2) = 105 pairs would dilute the signal and answer a different question.

In [ ]:
import numpy as np
# STATISTICAL TEST: Permutation Test (Lehto 2026, in review)
# Methodology: github.com/Chris-L78/cosmic-octaves-analysis
#
# Why permutation (not uniform-random Monte Carlo):
#   - Permutation preserves the exact measured log-lengths and only shuffles
#     label assignments, directly testing whether the OBSERVED values show
#     unusual alignment in the 7 predefined pairs.
#   - Uniform-random MC draws artificial points from a flat distribution,
#     destroying the natural clustering of real structures and testing a
#     different (weaker) null hypothesis.
#
# Why 7 fixed pairs (not all C(15,2)=105 pairs):
#   - The cosmic octave hypothesis predicts specific pairings (each rung has
#     one partner 10^24 larger). Testing all 105 pairs dilutes the signal
#     and tests a different question ("do any pairs land near 24?").

# 15 log10(L) values in canonical label order
logs = np.array([
    -15.08,   # Proton (RMS charge radius, CODATA 2022)
    -10.28,   # Atomic Orbital (Bohr radius, CODATA 2022)
     -7.96,   # Ribosome (half diameter 70S, BioNumbers)
     -6.00,   # Bacterium (half cell length, BioNumbers)
     -3.30,   # C. elegans (half body length, NCBI)
     -0.046,  # Human (half body height 1.8 m)
      3.00,   # City (coordination radius, West 2017)
      6.80,   # Earth (mean radius, IAU 2015)
      8.84,   # Sun (photosphere radius, IAU 2015)
     12.65,   # Solar System (Neptune orbit, IAU 2015)
     16.67,   # Open Cluster (half-mass radius ~1.5 pc)
     18.665,  # Local Bubble (median boundary, Pelgrims+ 2020)
     20.70,   # Milky Way (half stellar disk, Bland-Hawthorn 2016)
     23.84,   # Virgo Supercluster (half density-extent proxy)
     26.64    # Observable Universe (particle horizon, Planck 2018)
])

# 7 predefined canonical pairs (small_index, large_index)
pairs = [
    (0, 8),   # Proton -> Sun
    (1, 9),   # Atomic Orbital -> Solar System
    (2, 10),  # Ribosome -> Open Cluster
    (3, 11),  # Bacterium -> Local Bubble
    (4, 12),  # C. elegans -> Milky Way
    (5, 13),  # Human -> Local Supercluster
    (6, 14)   # City -> Observable Universe
]

def get_deviations(arr, delta=24.0):
    """Deviation of each pair's log-ratio from the ideal delta."""
    return np.array([abs((arr[j] - arr[i]) - delta) for i, j in pairs])

# Observed deviations
obs = get_deviations(logs)
obs_strong = int(np.sum(obs <= 0.2))

pair_names = [
    "Proton -> Sun", "Atomic -> Solar Sys", "Ribosome -> Open Cl",
    "Bacterium -> Local B", "C. elegans -> MW", "Human -> Virgo SC",
    "City -> Obs Universe"
]
print(f"\n7 Canonical Octave Pairs:")
for name, dev in zip(pair_names, obs):
    quality = ("PERFECT" if dev < 0.05 else "Excellent" if dev <= 0.2
               else "Very Good" if dev <= 0.7 else "Good" if dev <= 1.0 else "Fair")
    print(f"  {name:22s}  D = {dev:.3f}  ({quality})")

print(f"\nStrong matches (D <= 0.2): {obs_strong}/7")

# Permutation test: 200,000 trials, fixed seed for reproducibility
n_trials = 200000
rng = np.random.default_rng(42)
count = 0
for _ in range(n_trials):
    perm = rng.permutation(logs)
    if np.sum(get_deviations(perm) <= 0.2) >= obs_strong:
        count += 1

p_value = count / n_trials
print(f"\nPermutation p-value: {p_value} ({p_value*100:.4f}%)")
print(f"Successes: {count} out of {n_trials}")
print(f"Significance: ~3.9 sigma (one-sided normal equivalent)")

## Section VI: Local Bubble Population Estimate (Section 5.3)

Verification code for the manuscript Chapter 15 claim that the Milky Way contains ~10³ Local Bubble-scale cavities. Two independent methods are computed here, showing they converge on the same order of magnitude.

**Method 1 (Catalog-based):** Whole-sky HI shell catalogs (Ehlerová & Palouš 2013) identify 333 shells. Outer-Galaxy surveys at higher sensitivity (Suad et al. 2014) find 566 supershell candidates in 2 quadrants alone. Extrapolating with completeness corrections gives ~10³ shells of Local Bubble scale (radius ≥100 pc) Galaxy-wide.

**Method 2 (Steady-state from supernova rate):** Galactic SN rate ~2.84×10⁻² yr⁻¹ (Park et al. 2013), Local Bubble lifetime ~14 Myr (Breitschwerdt & de Avillez 2006), fraction of SNe carving Local Bubble-scale cavities ~5%, gives ~10³.

Both methods land at the same order of magnitude — supporting the C. elegans (959 cells) to Milky Way (~10³ cavities) octave-pair comparison.

In [ ]:
import math

# ----- Method 1: Catalog-based extrapolation -----
# Whole-sky HI shell catalogs identify ~300 shells (LAB survey, Ehlerova & Palous 2013).
# Higher-sensitivity outer-Galaxy surveys (Suad et al. 2014) find ~566 in just 2 quadrants.
# Restricting to Local Bubble scale (radius >= 100 pc) with completeness corrections,
# the order-of-magnitude estimate is ~10^3 cavities Galaxy-wide.
LAB_whole_galaxy = 333          # Ehlerova & Palous 2013, LAB HI survey
suad_outer_2quadrants = 566     # Suad et al. 2014, outer 2 of 4 Galactic quadrants
suad_full_galaxy_naive = suad_outer_2quadrants * 2  # naive 2x extrapolation
# Apply 50% restriction to LB-scale (radius >= 100 pc; smaller cavities excluded)
fraction_lb_scale = 0.50
catalog_estimate = suad_full_galaxy_naive * fraction_lb_scale

print("Method 1: Catalog-based extrapolation")
print(f"  LAB whole-galaxy direct count (lower bound): {LAB_whole_galaxy}")
print(f"  Suad et al. 2014 (outer 2 quadrants):         {suad_outer_2quadrants}")
print(f"  Naive linear extrapolation to full galaxy:    ~{suad_full_galaxy_naive}")
print(f"  Restricted to LB-scale cavities (r>=100 pc):  ~{int(catalog_estimate)}")
print(f"  Order of magnitude: 10^{math.log10(catalog_estimate):.1f}")

# ----- Method 2: Volume filling factor approach -----
# Sun et al. 2024 (MNRAS) simulations of Milky Way-like galaxies find that
# hot gas (T > 10^5.5 K) has a volume filling factor of ~12% in the disk.
# Dividing total hot-gas volume by typical Local Bubble volume gives an
# independent estimate of the LB-scale cavity count.
kpc_in_pc = 1000
disk_radius_pc = 15 * kpc_in_pc      # 15 kpc disk extent for hot gas
disk_height_pc = 0.4 * kpc_in_pc     # 400 pc scale height (conservative)
disk_volume_pc3 = math.pi * disk_radius_pc**2 * disk_height_pc

hot_gas_filling_factor = 0.12        # Sun et al. 2024
hot_gas_volume_pc3 = disk_volume_pc3 * hot_gas_filling_factor

LB_radius_pc = 150                   # Local Bubble characteristic radius
LB_volume_pc3 = (4/3) * math.pi * LB_radius_pc**3

filling_factor_estimate = hot_gas_volume_pc3 / LB_volume_pc3

print()
print("Method 2: Volume filling factor approach")
print(f"  Disk volume (15 kpc x 400 pc):           {disk_volume_pc3:.2e} pc^3")
print(f"  Hot gas filling factor (Sun et al. 2024): {hot_gas_filling_factor}")
print(f"  Total hot gas volume:                     {hot_gas_volume_pc3:.2e} pc^3")
print(f"  Typical Local Bubble volume (r=150 pc):   {LB_volume_pc3:.2e} pc^3")
print(f"  Cavity count = hot vol / typical LB vol: ~{int(filling_factor_estimate)}")
print(f"  Order of magnitude: 10^{math.log10(filling_factor_estimate):.1f}")

# ----- Convergence -----
print()
print("----- Convergence -----")
print(f"  Method 1 (catalog):          ~{int(catalog_estimate):,}  (order 10^{math.log10(catalog_estimate):.1f})")
print(f"  Method 2 (filling factor):   ~{int(filling_factor_estimate):,}  (order 10^{math.log10(filling_factor_estimate):.1f})")
print()
print("Both methods land at order 10^3 with factor-of-a-few uncertainty.")
print("This is an order-of-magnitude estimate, not a precise count.")

# ----- C. elegans comparison -----
celegans_cells = 959   # Sulston et al. 1983, exact lineage count
print()
print(f"C. elegans somatic cells (exact):              {celegans_cells}")
print(f"Milky Way Local Bubble-scale cavities (~10^3): order {math.log10(catalog_estimate):.1f}")
print()
print("Octave-pair comparison: 10^3 vs 10^3, agreement at order-of-magnitude level.")
print("This is NOT a percent-level claim — both numbers carry real uncertainty.")
